In [1]:
from importlib.resources.readers import remove_duplicates
from typing import Optional, Callable
from chembl_webresource_client.new_client import new_client
import pandas as pd
from pandas.core.interchange.dataframe_protocol import DataFrame


def find_targets(query: str, organism="Homo sapiens", limit=None):
    t = new_client.target
    hits = t.search(query)
    rows = []
    for h in hits:
        if organism and h.get("organism") == organism:
            rows.append(
                {
                    "target_chembl_id": h["target_chembl_id"],
                    "pref_name": h.get("pref_name"),
                    "organism": h.get("organism"),
                    "target_type": h.get("target_type"),
                }
            )
            if limit and len(rows) >= limit:
                break
    return pd.DataFrame(rows)


print(find_targets("EGFR"))

C:\Users\reshv\miniconda3\envs\lead_craft_api\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


   target_chembl_id                                          pref_name  \
0     CHEMBL4523747                                        EGFR/PPP1CA   
1     CHEMBL5465557                                          CCN2-EGFR   
2         CHEMBL203                   Epidermal growth factor receptor   
3     CHEMBL4523680  Protein cereblon/Epidermal growth factor receptor   
4     CHEMBL2111431  Epidermal growth factor receptor and ErbB2 (HE...   
5     CHEMBL2363049                   Epidermal growth factor receptor   
6     CHEMBL4523998    von Hippel-Lindau disease tumor suppressor/EGFR   
7     CHEMBL4630723                          ErbB-2/ErbB-3 heterodimer   
8        CHEMBL1824            Receptor tyrosine-protein kinase erbB-2   
9        CHEMBL3009            Receptor tyrosine-protein kinase erbB-4   
10       CHEMBL5838            Receptor tyrosine-protein kinase erbB-3   
11    CHEMBL3137284  MER intracellular domain/EGFR extracellular do...   
12    CHEMBL4106134                   

In [2]:
targets = find_targets("EGFR")
print(targets.shape)

(17, 4)


In [3]:
import re
import math
from typing import Optional


def get_potencies_for_target(target_chembl_id: str,
                             types=("IC50", "Ki", "Kd", "EC50"),
                             ) -> list[dict[str, str]]:
    act = new_client.activity
    fields = [
        "activity_id", "assay_chembl_id", "assay_type", "assay_confidence_score",
        "molecule_chembl_id",
        "standard_type", "standard_value", "standard_units", "relation",
        "pchembl_value", "target_chembl_id"
    ]
    q = act.filter(
        target_chembl_id=target_chembl_id,
        standard_type__in=list(types)
    ).only(fields)
    data = list(q)
    return data


UNIT_FACTORS = {
    "m": 1.0,
    "mm": 1e-3,
    "um": 1e-6,
    "nm": 1e-9,
    "pm": 1e-12,
}


def standard_unit_convertor_to_pchembl(units: str, value: float) -> float:
    if value <= 0:
        raise ValueError("value must be positive")
    u = units.strip().lower()
    u = u.replace("µ", "u")
    u = re.sub(r"\s+", "", u)
    u = u.replace("mol/l", "").replace("molperl", "")

    if u in UNIT_FACTORS.keys():
        standard_value = UNIT_FACTORS[units] * value
        return - math.log10(standard_value)
    else:
        raise ValueError(f"Unknown unit {units}")


def pchembl_extractor(activity_entry: dict[str, str]) -> Optional[float]:
    if activity_entry.get("pchembl_value"):
        return float(activity_entry["pchembl_value"])
    if activity_entry.get("standard_value") and activity_entry["standard_value"].isnumeric():
        units = activity_entry.get("standard_units")
        if units:
            return standard_unit_convertor_to_pchembl(units=units,
                                                      value=float
                                                      (activity_entry["standard_value"])
                                                      )

    if activity_entry.get("value") and activity_entry["value"].isnumeric():
        units = activity_entry.get("units")
        if units:
            return standard_unit_convertor_to_pchembl(units=units,
                                                      value=float(activity_entry["value"])
                                                      )

    return None


def activities_formator(activities_full_df: pd.DataFrame) -> pd.DataFrame:
    if activities_full_df.empty:
        raise ValueError("No activity data found")
    activities_df = activities_full_df[["molecule_chembl_id",
                                        "activity_id",
                                        "assay_chembl_id",
                                        "assay_type",
                                        ]]

    activities_df["pchembl_value"] = [pchembl_extractor(activity_entry) for activity_entry in activities]
    return activities_df


# TODO add function converting list to pd + extract pchembl
# TODO add function cheking that value is numeric
# TODO add function to check if compound active/inactive from assay_type and pchembl
# TODO

# --- 4) (Optional) Attach canonical SMILES for molecules ---


def attach_smiles(df: pd.DataFrame) -> pd.DataFrame:
    mol = new_client.molecule
    ids = df["molecule_chembl_id"].dropna().unique().tolist()
    chunks = [ids[i:i + 100] for i in range(0, len(ids), 100)]
    id_to_smiles = {}
    for chunk in chunks:
        mres = mol.filter(molecule_chembl_id__in=chunk).only(
            ["molecule_chembl_id", "molecule_structures"]
        )
        for m in mres:
            smi = None
            if m.get("molecule_structures"):
                smi = m["molecule_structures"].get("canonical_smiles")
            id_to_smiles[m["molecule_chembl_id"]] = smi
    df = df.copy()
    df["canonical_smiles"] = df["molecule_chembl_id"].map(id_to_smiles)
    return df

In [4]:
a = get_potencies_for_target("CHEMBL203")
print(type(a))

<class 'list'>


In [5]:
print(a[0])

{'activity_id': 32260, 'assay_chembl_id': 'CHEMBL674637', 'assay_type': 'B', 'molecule_chembl_id': 'CHEMBL68920', 'pchembl_value': '7.39', 'relation': '=', 'standard_type': 'IC50', 'standard_units': 'nM', 'standard_value': '41.0', 'target_chembl_id': 'CHEMBL203', 'type': 'IC50', 'units': 'uM', 'value': '0.041'}


In [6]:
import numbers

print(len(a))
print(len([i for i in a if "pchembl_value" in i.keys()]))
b = [i for i in a if isinstance(i["pchembl_value"], str)]
print(b[0:3])

30166
30166
[{'activity_id': 32260, 'assay_chembl_id': 'CHEMBL674637', 'assay_type': 'B', 'molecule_chembl_id': 'CHEMBL68920', 'pchembl_value': '7.39', 'relation': '=', 'standard_type': 'IC50', 'standard_units': 'nM', 'standard_value': '41.0', 'target_chembl_id': 'CHEMBL203', 'type': 'IC50', 'units': 'uM', 'value': '0.041'}, {'activity_id': 32263, 'assay_chembl_id': 'CHEMBL621151', 'assay_type': 'F', 'molecule_chembl_id': 'CHEMBL68920', 'pchembl_value': '6.52', 'relation': '=', 'standard_type': 'IC50', 'standard_units': 'nM', 'standard_value': '300.0', 'target_chembl_id': 'CHEMBL203', 'type': 'IC50', 'units': 'uM', 'value': '0.3'}, {'activity_id': 32265, 'assay_chembl_id': 'CHEMBL615325', 'assay_type': 'F', 'molecule_chembl_id': 'CHEMBL68920', 'pchembl_value': '5.11', 'relation': '=', 'standard_type': 'IC50', 'standard_units': 'nM', 'standard_value': '7820.0', 'target_chembl_id': 'CHEMBL203', 'type': 'IC50', 'units': 'uM', 'value': '7.82'}]


In [10]:
def activities_to_dataframe(activities: list[dict[str, Optional[str]]]) -> pd.DataFrame:
    if not activities:
        raise ValueError("No activities found")
    activities_df = pd.DataFrame(activities, index=None)
    activities_df = activities_df[["molecule_chembl_id",
                                   "activity_id",
                                   "assay_chembl_id",
                                   "assay_type",
                                   "relation"
                                   ]]
    activities_df["pchembl_value"] = [pchembl_extractor(activity_entry) for activity_entry in activities]
    return activities_df


def add_activity_classification():
    pass


my_list = [
    {'activity_id': 32260, 'assay_chembl_id': 'CHEMBL674637', 'assay_type': 'B', 'molecule_chembl_id': 'CHEMBL68920',
     'pchembl_value': '7.39', 'relation': '=', 'standard_type': 'IC50', 'standard_units': 'nM',
     'standard_value': '41.0', 'target_chembl_id': 'CHEMBL203', 'type': 'IC50', 'units': 'uM', 'value': '0.041'},
    {'activity_id': 32263, 'assay_chembl_id': 'CHEMBL621151', 'assay_type': 'F', 'molecule_chembl_id': 'CHEMBL68920',
     'pchembl_value': '6.52', 'relation': '=', 'standard_type': 'IC50', 'standard_units': 'nM',
     'standard_value': '300.0', 'target_chembl_id': 'CHEMBL203', 'type': 'IC50', 'units': 'uM', 'value': '0.3'},
    {'activity_id': 32265, 'assay_chembl_id': 'CHEMBL615325', 'assay_type': 'F', 'molecule_chembl_id': 'CHEMBL68920',
     'pchembl_value': '5.11', 'relation': '=', 'standard_type': 'IC50', 'standard_units': 'nM',
     'standard_value': '7820.0', 'target_chembl_id': 'CHEMBL203', 'type': 'IC50', 'units': 'uM', 'value': '7.82'}]

print(activities_to_dataframe(my_list))
print(activities_to_dataframe(a).head())


  molecule_chembl_id  activity_id assay_chembl_id assay_type relation  \
0        CHEMBL68920        32260    CHEMBL674637          B        =   
1        CHEMBL68920        32263    CHEMBL621151          F        =   
2        CHEMBL68920        32265    CHEMBL615325          F        =   

   pchembl_value  
0           7.39  
1           6.52  
2           5.11  
  molecule_chembl_id  activity_id assay_chembl_id assay_type relation  \
0        CHEMBL68920        32260    CHEMBL674637          B        =   
1        CHEMBL68920        32263    CHEMBL621151          F        =   
2        CHEMBL68920        32265    CHEMBL615325          F        =   
3        CHEMBL69960        32267    CHEMBL674637          B        =   
4        CHEMBL69960        32270    CHEMBL621151          F        =   

   pchembl_value  
0           7.39  
1           6.52  
2           5.11  
3           6.77  
4           7.40  


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
import pandas as pd
from typing import Optional, Iterable, Tuple


def morgan_finger_prints_from_smiles(smiles: str):  # TODO: add return value
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048) if mol else None


def add_mols_and_fps(
        df: pd.DataFrame,
        smiles_col: str = "canonical_smiles",
        mol_col: str = "mol",
        fp_col: str = "ecfp4"
) -> pd.DataFrame:
    mols = []
    fps = []
    for smi in df[smiles_col].astype(str):
        m = Chem.MolFromSmiles(smi)
        mols.append(m)
        fps.append(AllChem.GetMorganFingerprintAsBitVect(m, radius=2, nBits=2048) if m else None)
    out = df.copy()
    out[mol_col] = mols
    out[fp_col] = fps
    out = out[out[mol_col].notna()].reset_index(drop=True)
    return out


def tanimoto_search(
        query_smiles: str,
        df: pd.DataFrame,
        fp_col: str = "ecfp4",
        top_k: int = 20,
        min_sim: float = 0.0
) -> pd.DataFrame:
    qm = Chem.MolFromSmiles(query_smiles)
    if qm is None:
        raise ValueError("Invalid query SMILES")
    qfp = AllChem.GetMorganFingerprintAsBitVect(qm, radius=2, nBits=2048)

    sims = []
    fps = df[fp_col].tolist()
    for fp in fps:
        if fp is None:
            sims.append(float("nan"))
        else:
            sims.append(DataStructs.TanimotoSimilarity(qfp, fp))

    res = df.copy()
    res["tanimoto"] = sims
    res = res[pd.to_numeric(res["tanimoto"], errors="coerce").notna()]
    if min_sim > 0:
        res = res[res["tanimoto"] >= min_sim]
    return res.sort_values("tanimoto", ascending=False).head(top_k).reset_index(drop=True)


def substructure_search(
        query: str,
        df: pd.DataFrame,
        mol_col: str = "mol",
        as_smarts: bool = False
) -> pd.DataFrame:
    q = Chem.MolFromSmarts(query) if as_smarts else Chem.MolFromSmiles(query)
    if q is None:
        raise ValueError("Invalid query pattern")
    mask = df[mol_col].apply(lambda m: m.HasSubstructMatch(q) if m else False)
    return df[mask].reset_index(drop=True)

In [4]:
from rdkit import Chem
from rdkit.Chem import DataStructs, rdMolDescriptors

m1 = Chem.MolFromSmiles("c1ccc2c(c1)c(c(oc2=O)OCCSC(=N)N)Cl")
m2 = Chem.MolFromSmiles("COc1cc2ccc(cc2c(=O)o1)NC(=N)N")
# finger_print_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2,
#                                                              fpSize=2048,
#                                                              countSimulation=True)
fp1 = rdMolDescriptors.GetMACCSKeysFingerprint(m1)
fp2 = rdMolDescriptors.GetMACCSKeysFingerprint(m2)

sim = DataStructs.TanimotoSimilarity(fp1, fp2)  # float 0..1
print(f"{sim:.3f}")

0.589


In [5]:
from rdkit.Chem import rdMolDescriptors

fp1 = rdMolDescriptors.GetMACCSKeysFingerprint(m1)
fp2 = rdMolDescriptors.GetMACCSKeysFingerprint(m2)
print("Tanimoto(MACCS):", DataStructs.TanimotoSimilarity(fp1, fp2))

Tanimoto(MACCS): 0.5892857142857143


In [9]:
from typing import Optional


def similarity_search(
        mol_search: Chem.rdchem.Mol,
        smiles_df: str) -> Optional[float]:
    if not mol_search:
        raise ValueError("Invalid search SMILES")
    mol_df = Chem.MolFromSmiles(smiles_df)
    if not mol_df:
        return None
    fp_search = rdMolDescriptors.GetMACCSKeysFingerprint(mol_search)
    fp_df = rdMolDescriptors.GetMACCSKeysFingerprint(mol_df)
    return DataStructs.TanimotoSimilarity(fp_search, fp_df)


In [10]:
print(similarity_search(Chem.MolFromSmiles("c1ccc2c(c1)c(c(oc2=O)OCCSC(=N)N)Cl"), "COc1cc2ccc(cc2c(=O)o1)NC(=N)N"))

0.5892857142857143


In [ ]:
from functools import partial


def similarity_search_for_query(smiles: str) -> Callable[[str], Optional[float]]:
    mol_search = Chem.MolFromSmiles(smiles)
    if not mol_search:
        raise ValueError("Invalid SMILES")
    return partial(similarity_search, mol_search)


def similarity_column_generation(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["similarity"] = df["smiles"].apply(similarity_search, axis=1)
    return df

In [ ]:
def fetch_smiles(mol_ids: pd.Series, batch: int = 100) -> list[str]:
    mol = new_client.molecule
    rows = []
    ids = mol_ids.dropna().unique().tolist()
    for i in range(0, len(ids), batch):
        chunk = ids[i:i + batch]
        for m in mol.filter(molecule_chembl_id__in=chunk).only(
                ["molecule_chembl_id", "molecule_structures"]
        ):
            smi = None
            if m.get("molecule_structures"):
                smi = m["molecule_structures"].get("canonical_smiles")
            rows.append({"molecule_chembl_id": m["molecule_chembl_id"], "canonical_smiles": smi})
    df = pd.DataFrame(rows)
    return df.dropna(subset=["canonical_smiles"])


In [3]:
from typing import Union, List
from chembl_webresource_client.new_client import new_client
import pandas as pd


def get_activities_for_target(target_chembl_id: str,
                              types=("IC50", "Ki", "Kd", "EC50"),
                              as_dataframe: bool = True,
                              ) -> Union[pd.DataFrame, List[dict]]:
    act = new_client.activity
    fields = [
        "activity_id", "assay_chembl_id", "assay_type", "assay_confidence_score",
        "molecule_chembl_id",
        "standard_type", "standard_value", "standard_units", "relation",
        "pchembl_value", "target_chembl_id"
    ]
    q = act.filter(
        target_chembl_id=target_chembl_id,
        standard_type__in=list(types)
    ).only(fields)
    data = list(q)

    if as_dataframe:
        if not data:
            return pd.DataFrame(columns=fields)
        df = pd.DataFrame(data)
        return df
    return data


def combine_activities_for_targets(target_ids: List[str],
                                   types=("IC50", "Ki", "Kd", "EC50"),
                                   ) -> pd.DataFrame:
    all_activities = []

    for i, target_id in enumerate(target_ids):
        activities_df = get_activities_for_target(target_id, types=types, as_dataframe=True)
        if not activities_df.empty:
            all_activities.append(activities_df)

    if not all_activities:
        raise ValueError("No activities found")
    combined_df = pd.concat(all_activities, ignore_index=True)
    return combined_df

In [4]:
df = combine_activities_for_targets(["CHEMBL203", "CHEMBL203", "CHEMBL4523747"])

In [5]:
print(df.shape)

(60332, 13)


In [11]:
# print(df["standard_type"].value_counts())
print(df["assay_type"].value_counts())

assay_type
B    55622
F     3114
A     1550
T       46
Name: count, dtype: int64


In [12]:
def get_activities_for_target(target_chembl_id: str,
                              types=("IC50", "Ki", "Kd", "EC50"),
                              as_dataframe: bool = True,
                              ) -> Union[pd.DataFrame, List[dict]]:
    act = new_client.activity
    fields = [
        "activity_id", "assay_chembl_id", "assay_type", "assay_confidence_score",
        "molecule_chembl_id",
        "standard_type", "standard_value", "standard_units", "relation",
        "pchembl_value", "target_chembl_id", "target_type"
    ]
    q = act.filter(
        target_chembl_id=target_chembl_id,
        standard_type__in=list(types)
    ).only(fields)
    data = list(q)

    if as_dataframe:
        if not data:
            return pd.DataFrame(columns=fields)
        df = pd.DataFrame(data)
        return df
    return data


b = get_activities_for_target(target_chembl_id="CHEMBL203")

In [19]:
def assay_info_extractor(activities_df: pd.DataFrame) -> dict[str, str]:
    assays = activities_df["assay_chembl_id"].dropna().unique().tolist()
    assay_info = list(new_client.assay.filter(assay_chembl_id__in=assays)
                      .only(
        ["assay_chembl_id", "assay_format", "assay_cell_type", "assay_organism", "bao_format", "bao_label"]))
    return assay_info

def assay_type_extractor(assay_info: dict[str, str]) -> Optinal[str]:
    fmt = (assay_info.get("bao_label")).lower()
    if fmt:
        if fmt == "biochemical": return "biochemical"
        if fmt == "cell-based":  return "cellular"
        if fmt == "organism-based": return "organism"
    if assay_info.get("assay_cell_type"): return "cellular"
    return None




In [20]:
print(assay_info_extractor(df)[0:3])

[{'assay_cell_type': 'NIH3T3', 'assay_chembl_id': 'CHEMBL615325', 'assay_organism': 'Homo sapiens', 'bao_format': 'BAO_0000219', 'bao_label': 'cell-based format'}, {'assay_cell_type': 'NIH3T3', 'assay_chembl_id': 'CHEMBL615490', 'assay_organism': 'Homo sapiens', 'bao_format': 'BAO_0000219', 'bao_label': 'cell-based format'}, {'assay_cell_type': 'A-431', 'assay_chembl_id': 'CHEMBL621022', 'assay_organism': None, 'bao_format': 'BAO_0000219', 'bao_label': 'cell-based format'}]
